In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy.special as special
import scipy.optimize as optimize
from astropy.table import Table
from astropy.io import fits

In [ ]:
WPIXSCALE = 0.059 # arcsec
ZPIXSCALE = 3. * WPIXSCALE
N, Nw, Nz = 120, 18, 12
R = np.sqrt((ZPIXSCALE * (3 + Nz / 2))**2 + (WPIXSCALE * (3 + Nw / 2))**2) + 0.05
#Set the R parameter to something a bit bigger than half the padded scan diagonal

In [ ]:
ws = (WPIXSCALE / 2) * (np.arange(2 * (Nw + 6)) - 0.5 * (2 * (Nw + 6) - 1.))
zs = (WPIXSCALE / 2) * (np.arange(6 * (Nz + 6)) - 0.5 * (6 * (Nz + 6) - 1.))
zgrid, wgrid = np.meshgrid(zs, ws) #These are the padded real space control points for the W calculation
rgrid_wz = np.sqrt(wgrid**2 + zgrid**2)
phigrid_wz = np.atan2(zgrid, wgrid)

In [ ]:
def get_pixel_average(arr):
    """
    ## input: 
    - `arr`: a (2 * (Nw + 6)) x (6 * (Nz + 6)) x n array of evaluated control points
    
    ## output:
    - a (Nw + 6) x (Nz + 6) x n array corresponding to the pixel average of the control points
    """
    meanz = np.mean(arr.reshape((arr.shape[0], np.int32(arr.shape[1] / 6), 6, -1)), 2)
    return np.mean(meanz.reshape((np.int32(arr.shape[0] / 2), 2, np.int32(arr.shape[1] / 6), -1)), 1)

In [ ]:
"""
plt.figure(figsize=(27, 9))
wpixcoords, zpixcoords = wgrid / WPIXSCALE, zgrid / ZPIXSCALE #Plot in pixel space
plt.scatter(wpixcoords, zpixcoords, s=0.3, c='k')
plt.vlines(np.arange(Nw+1) - 0.5*Nw, ymin=0-0.5*Nz, ymax=0.5*Nz, linewidth=0.5, color='k')
plt.hlines(np.arange(Nz+1) - 0.5*Nz, xmin=0-0.5*Nw, xmax=0.5*Nw, linewidth=0.5, color='k')
plt.gca().set_aspect(3.)
"""

In [ ]:
"""
wpixavg, zpixavg = get_pixel_average(wpixcoords), get_pixel_average(zpixcoords)
plt.figure(figsize=(27, 9))
plt.scatter(wpixavg, zpixavg, s=0.3, c='k')
plt.vlines(np.arange(Nw+1) - 0.5*Nw, ymin=0-0.5*Nz, ymax=0.5*Nz, linewidth=0.5, color='k')
plt.hlines(np.arange(Nz+1) - 0.5*Nz, xmin=0-0.5*Nw, xmax=0.5*Nw, linewidth=0.5, color='k')
plt.gca().set_aspect(3.)
"""

In [ ]:
def Jzeros(Nm, Nn):
    """
    ## input: 
    - `Nm`: a positive integer
    - `Nn`: a positive integer
    
    ## output:
    - a (2 * Nm - 1) x Nn array, each entry corresponding to the nth zero of the mth Bessel function, symmetric around 0 in m
    """
    Jzeros = np.zeros([2 * Nm - 1, Nn])
    for m in np.arange(Nm):
        col = special.jn_zeros(m, Nn)
        Jzeros[Nm - 1 + m] = col
        Jzeros[Nm - 1 - m] = col
    return Jzeros

def besselize(Nm, Nn, rs, phis, R):
    ms_rav = np.ravel(np.ones(Nn)[None,:] * (np.arange(2 * Nm - 1)- Nm + 1)[:,None])
    Jzeros_rav = np.ravel(Jzeros(Nm, Nn))
    samples = special.jv(np.abs(ms_rav[None,:]), Jzeros_rav[None,:] * rs[:,:,None] / R) * np.exp(1j * ms_rav[None,:] * phis[:,:,None])
    return samples

def W(Nm, Nn, rs, phis, R):
    samples = besselize(Nm, Nn, rs, phis, R)
    pixs = get_pixel_average(samples)
    W_fft = (np.fft.fft2(pixs, axes=(0, 1))).reshape(-1, (2 * Nm - 1) * Nn)
    return W_fft

In [ ]:
Nm = 12
Nn = 15

W_test = W(Nm, Nn, rgrid_wz, phigrid_wz, R)

In [ ]:
"""
fig, axs = plt.subplots(9, 10, figsize=(12, 24))
cscale = 3 / np.sqrt(11*10)
for n in range(10):
    for m in range(9):
        a_mns = np.zeros([2*Nm-1, Nn])
        a_mns[Nm - 4 + m - 1, n] = 1
        a_mns = np.ravel(a_mns)
        pixtest = np.fft.ifft2(W_test.reshape(Nw + 6, Nz + 6, -1) @ a_mns)
        axs[m][n].imshow(np.real(pixtest.T), origin="lower", interpolation="nearest", cmap="RdBu", aspect=3, vmin=-cscale, vmax=cscale)
        axs[m][n].axis("off")
plt.tight_layout()
#fig.savefig("bessels_re.png")
plt.show()
"""

In [ ]:
"""
fig, axs = plt.subplots(9, 10, figsize=(12, 24))
cscale = 3 / np.sqrt(11*10)
for n in range(10):
    for m in range(9):
        a_mns = np.zeros([2*Nm-1, Nn])
        a_mns[Nm - 4 + m - 1, n] = 1
        a_mns = np.ravel(a_mns)
        pixtest = np.fft.ifft2(W_test.reshape(Nw + 6, Nz + 6, -1) @ a_mns)
        axs[m][n].imshow(np.imag(pixtest.T), origin="lower", interpolation="nearest", cmap="RdBu", aspect=3, vmin=-cscale, vmax=cscale)
        axs[m][n].axis("off")
plt.tight_layout()
#fig.savefig("bessels_im.png")
plt.show()
"""

In [ ]:
xs = (WPIXSCALE / 2) * (np.arange(1 + np.int32(2 * R / (WPIXSCALE / 2))) - 0.5 * (1 + np.int32(2 * R / (WPIXSCALE / 2)) - 1.))
ygrid, xgrid = np.meshgrid(xs, xs)
rgrid_sc = np.sqrt(xgrid**2 + ygrid**2)
circle_mask = rgrid_sc > R
phigrid_sc = np.atan2(ygrid, xgrid)

def make_circle(arr):
    arr[circle_mask] = np.nan
    return arr

In [ ]:
plt.figure(figsize=(8, 8))
angles_test = 2*np.pi*np.arange(1e6) / 1e6
plt.plot(R*np.cos(angles_test), R*np.sin(angles_test), linewidth=0.4, c='k')
plt.vlines(0, ymin=-R, ymax=R, linewidth=0.4, color='k')
plt.hlines(0, xmin=-R, xmax=R, linewidth=0.4, color='k')
plt.scatter(make_circle(xgrid), make_circle(ygrid), s=0.3, c='k')
plt.gca().set_aspect(1.)

In [ ]:
scene_op = besselize(Nm, Nn, rgrid_sc, phigrid_sc, R)

In [ ]:
"""
fig, axs = plt.subplots(12, 9, figsize=(12, 18))
cscale = 2000 / np.sqrt(np.pi * R**2 / (WPIXSCALE / 2)**2)
for k in range(12):
    for l in range(9):
        y_kls = np.zeros([Nw + 6, Nz + 6])
        y_kls[k, l] = 1
        y_kls = np.ravel(y_kls)
        imtest = make_circle(np.real(scene_op@((y_kls.T@W_test).T)).T)
        axs[k][l].imshow(imtest, origin="lower", interpolation="nearest", cmap="RdBu", vmax=cscale, vmin=-cscale)
        axs[k][l].axis("off")
plt.tight_layout()
#fig.savefig("scene_re.png")
plt.show()
"""

In [ ]:
"""
fig, axs = plt.subplots(12, 9, figsize=(12, 18))
cscale = 2000 / np.sqrt(np.pi * R**2 / (WPIXSCALE / 2)**2)
for k in range(12):
    for l in range(9):
        y_kls = np.zeros([Nw + 6, Nz + 6])
        y_kls[k, l] = 1
        y_kls = np.ravel(y_kls)
        imtest = make_circle(np.imag(scene_op@((y_kls.T@W_test).T)).T)
        axs[k][l].imshow(imtest, origin="lower", interpolation="nearest", cmap="RdBu", vmax=cscale, vmin=-cscale)
        axs[k][l].axis("off")
plt.tight_layout()
#fig.savefig("scene_im.png")
plt.show()
"""

In [ ]:
def X_rotshift(W, psis, offsets, Nw, Nz, Nm, Nn):
    rot_vec = np.exp(1j * psis[:,None] * np.ravel(np.ones(Nn)[None,:] * (np.arange(2 * Nm - 1) - Nm + 1)[:,None])[None,:])
    rot_mat = (rot_vec[:,None,:] * np.ones(Nw * Nz)[None,:,None]).reshape(N * Nw * Nz, -1)
    dispw_vec = np.exp(1j * -offsets[:,0][:,None] * WPIXSCALE * (np.fft.fftfreq(Nw, WPIXSCALE) * 2*np.pi)[None,:])
    dispz_vec = np.exp(1j * -offsets[:,1][:,None] * ZPIXSCALE * (np.fft.fftfreq(Nz, ZPIXSCALE) * 2*np.pi)[None,:])
    disp_vec_mat = dispw_vec[:,:,None] * dispz_vec[:,None,:]
    disp_vec_mat = np.ravel(disp_vec_mat)[:,None]
    W_big = np.reshape(W[None,:,:] * np.ones(N)[:,None,None], (N * Nw * Nz, -1))
    return disp_vec_mat * W_big * rot_mat

In [ ]:
def ell_ker(sigma, Nw, Nz):
    ells = np.fft.fftfreq(Nz, ZPIXSCALE) * 2*np.pi
    ell_filter = np.exp(-ells**2 / (2*sigma**2))
    return np.ravel(np.ravel(np.ones(Nw)[:,None] * ell_filter[None,:])[None,:] * np.ones(N)[:,None])

In [ ]:
def X_des(X_rotshift, ell_ker):
    return np.fft.ifft2((X_rotshift * ell_ker[:,None]).reshape(N, Nw + 6, Nz + 6, -1), axes=(1, 2))[:,3:-3,3:-3,:].reshape(N * Nw * Nz, -1)

In [ ]:
C_inv = np.eye(N * Nw * Nz) #inverse covariance matrix

In [ ]:
def solve(X, Y):
   # return np.linalg.lstsq(C_inv @ X, C_inv @ Y)[0]
    return np.linalg.lstsq(X, Y)[0]

In [ ]:
def chi_squared_sigma(sigma, Y, X_rotshift, a):
    guess = X_des(X_rotshift, ell_ker(sigma, Nw + 6, Nz + 6)) @ a
    #return ((Y - guess).conj().T @ C_inv @ (Y - guess)).real
    return ((Y - guess).conj().T @ (Y - guess)).real

In [ ]:
fn = "data_00019.fits"
with fits.open(fn) as hdul:
    Y = hdul[0].data
    X = hdul["x"].data
    truth = hdul["true"].data

def rotscale(psi):
    cp, sp = np.cos(psi), np.sin(psi)
    return np.array([[cp / WPIXSCALE, sp / WPIXSCALE], [-sp / ZPIXSCALE, cp / ZPIXSCALE]])

foo = np.array([1., 0.]) # arcsec
w0, z0 = 0.5 * (Nw - 1.), 0.5 * (Nz - 1.)
for j in range(2):
    plt.imshow(Y[j].T, origin="lower", interpolation="nearest", cmap="gray", aspect=3.)
    dw, dz = X["wz_offset"][j]
    plt.scatter(w0 + dw, z0 + dz, marker="x", c="r", s=50, lw=2)
    bar = rotscale(X["scan_angle"][j]) @ foo
    plt.plot([w0 + dw, w0 + dw + bar[0]], [z0 + dz, z0 + dz + bar[1]], "r-", lw=0.5)
    plt.xlim(-0.5, Nw - 0.5)
    plt.ylim(-0.5, Nz - 0.5)
    plt.show()

psis = X["scan_angle"]
offsets = X["wz_offset"]
Y_eq = np.ravel(Y)
true_loc = truth["xy"]

In [ ]:
X_design_300 = X_des(X_rotshift(W_test, psis, offsets, Nw + 6, Nz + 6, Nm, Nn), ell_ker(300, Nw + 6, Nz + 6))
a_test_300 = solve(X_design_300, Y_eq)

plt.imshow(make_circle(np.real(scene_op @ a_test_300)).T,
           extent = [-R, R, -R, R], origin="lower", interpolation="nearest", cmap="gray")
plt.scatter(true_loc[:,0], true_loc[:,1], marker="x", c="r", s=50, lw=2)
plt.tight_layout()
plt.show()

scans_300 = ((X_design_300 @ a_test_300).reshape(N, Nw, Nz)).real

for j in range(10):
    fig, ax = plt.subplots(1, 2)
    ax[0].imshow(Y[j].T, origin="lower", interpolation="nearest", cmap="gray", aspect=3.)
    ax[1].imshow(scans_300[j,:,:].T, origin="lower", interpolation="nearest", cmap="gray", aspect=3.)
    dw, dz = X["wz_offset"][j]
    ax[0].scatter(w0 + dw, z0 + dz, marker="x", c="r", s=50, lw=2)
    ax[1].scatter(w0 + dw, z0 + dz, marker="x", c="r", s=50, lw=2)
    bar = rotscale(X["scan_angle"][j]) @ foo
    ax[0].plot([w0 + dw, w0 + dw + bar[0]], [z0 + dz, z0 + dz + bar[1]], "r-", lw=0.5)
    ax[1].plot([w0 + dw, w0 + dw + bar[0]], [z0 + dz, z0 + dz + bar[1]], "r-", lw=0.5)
    ax[0].set_xlim(-0.5, Nw - 0.5)
    ax[0].set_ylim(-0.5, Nz - 0.5)
    ax[1].set_xlim(-0.5, Nw - 0.5)
    ax[1].set_ylim(-0.5, Nz - 0.5)
    plt.show()

In [ ]:
sigma = 6.43
X_rs = X_rotshift(W_test, psis, offsets, Nw + 6, Nz + 6, Nm, Nn)
for i in range(5):
    a = solve(X_des(X_rs, ell_ker(sigma, Nw + 6, Nz + 6)), Y_eq)
    sigma = optimize.minimize_scalar(fun=lambda sigma: chi_squared_sigma(sigma, Y_eq, X_rs, a)).x
    print(sigma)

In [ ]:
sigma_opt = sigma
print(sigma_opt)

In [ ]:
X_design_opt = X_des(X_rotshift(W_test, psis, offsets, Nw + 6, Nz + 6, Nm, Nn), ell_ker(sigma_opt, Nw + 6, Nz + 6))
a_test_opt = solve(X_design_opt, Y_eq)
scene_opt = make_circle((scene_op @ a_test_opt).real)

scans_opt = ((X_design_opt @ a_test_opt).reshape(N, Nw, Nz)).real
vmins = np.zeros(N)
vmaxs = np.zeros(N)

for j in range(N):
    vmins[j] = np.nanmin(scans_opt[j,:,:])
    vmaxs[j] = np.nanmax(scans_opt[j,:,:])
    """
    fig, ax = plt.subplots(1, 2)
    ax[0].imshow(Y[j].T, origin="lower", interpolation="nearest", cmap="gray", aspect=3.)
    ax[1].imshow(scans_opt[j,:,:].T, origin="lower", interpolation="nearest", cmap="gray", aspect=3.)
    dw, dz = X["wz_offset"][j]
    ax[0].scatter(w0 + dw, z0 + dz, marker="x", c="r", s=50, lw=2)
    ax[1].scatter(w0 + dw, z0 + dz, marker="x", c="r", s=50, lw=2)
    bar = rotscale(X["scan_angle"][j]) @ foo
    ax[0].plot([w0 + dw, w0 + dw + bar[0]], [z0 + dz, z0 + dz + bar[1]], "r-", lw=0.5)
    ax[1].plot([w0 + dw, w0 + dw + bar[0]], [z0 + dz, z0 + dz + bar[1]], "r-", lw=0.5)
    ax[0].set_xlim(-0.5, Nw - 0.5)
    ax[0].set_ylim(-0.5, Nz - 0.5)
    ax[1].set_xlim(-0.5, Nw - 0.5)
    ax[1].set_ylim(-0.5, Nz - 0.5)
    plt.show()
    """

vmins_avg = np.mean(vmins)
vmaxs_avg = np.mean(vmaxs)

plt.imshow(scene_opt.T, extent = [-R, R, -R, R], origin="lower", interpolation="nearest", cmap="gray", vmin=vmins_avg, vmax=vmaxs_avg)
plt.scatter(true_loc[:,0], true_loc[:,1], marker="x", c="r", s=50, lw=2)
plt.tight_layout()
plt.show()

In [ ]:
fn = "data_00003.fits"
with fits.open(fn) as hdul:
    Y3 = hdul[0].data
    X3 = hdul["x"].data
    truth3 = hdul["true"].data

psis3 = X3["scan_angle"]
offsets3 = X3["wz_offset"]
Y_eq3 = np.ravel(Y3)
true_loc3 = truth3["xy"]

X_design_3 = X_des(X_rotshift(W_test, psis3, offsets3, Nw + 6, Nz + 6, Nm, Nn), ell_ker(sigma_opt, Nw + 6, Nz + 6))
a_test_3 = solve(X_design_3, Y_eq3)
scene_3 = make_circle((scene_op @ a_test_3).real)
scans_3 = ((X_design_3 @ a_test_3).reshape(N, Nw, Nz)).real

vmins = np.zeros(N)
vmaxs = np.zeros(N)

for j in range(N):
    vmins[j] = np.nanmin(scans_3[j,:,:])
    vmaxs[j] = np.nanmax(scans_3[j,:,:])

vmins_avg = np.mean(vmins)
vmaxs_avg = np.mean(vmaxs)

plt.imshow(scene_3.T, extent = [-R, R, -R, R], origin="lower", interpolation="nearest", cmap="gray", vmin=vmins_avg, vmax=vmaxs_avg)
plt.scatter(true_loc3[:,0], true_loc3[:,1], marker="x", c="r", s=50, lw=2)
plt.tight_layout()
plt.show()

In [ ]:
fn = "data_00004.fits"
with fits.open(fn) as hdul:
    Y4 = hdul[0].data
    X4 = hdul["x"].data
    truth4 = hdul["true"].data

psis4 = X4["scan_angle"]
offsets4 = X4["wz_offset"]
Y_eq4 = np.ravel(Y4)
true_loc4 = truth4["xy"]

X_design_4 = X_des(X_rotshift(W_test, psis4, offsets4, Nw + 6, Nz + 6, Nm, Nn), ell_ker(sigma_opt, Nw + 6, Nz + 6))
a_test_4 = solve(X_design_4, Y_eq4)

scene_4 = make_circle((scene_op @ a_test_4).real)
scans_4 = ((X_design_4 @ a_test_4).reshape(N, Nw, Nz)).real

vmins = np.zeros(N)
vmaxs = np.zeros(N)

for j in range(N):
    vmins[j] = np.nanmin(scans_4[j,:,:])
    vmaxs[j] = np.nanmax(scans_4[j,:,:])

vmins_avg = np.mean(vmins)
vmaxs_avg = np.mean(vmaxs)

plt.imshow(scene_4.T, extent = [-R, R, -R, R], origin="lower", interpolation="nearest", cmap="gray", vmin=vmins_avg, vmax=vmaxs_avg)
plt.scatter(true_loc4[:,0], true_loc4[:,1], marker="x", c="r", s=50, lw=2)
plt.tight_layout()
plt.show()